# M3 L3: Embeddings and Model Selection

**Build a model registry, drift eval, and dual-index migration sketch from scratch.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

An **embedding** is what an AI gives you when you ask it "turn this sentence into a list of numbers that captures the meaning." Similar sentences get similar lists.

The trick: **different AI models give you different lists for the same sentence.** You can't just average them or mix them up. Switching from one model to another is a real engineering project — like switching from kilometers to miles in a navigation system while cars are still driving.

## The 5 things to know

1. **Embeddings are model-specific** — v1 and v2 of the same text live in different spaces; don't mix.
2. **L2-normalize at ingest** — cosine via inner product is 12× faster and gives the same ranking.
3. **Drift eval** — measure mean cosine between same-text vectors from two models; below 0.7 means the spaces are too different.
4. **Dual-index migration** — build v2 in the background, route 1% → 10% → 50% → 100% traffic; never one-shot.
5. **The golden set is the contract** — change the eval, you change what's being measured.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm03-embeddings' / 'L3-embeddings-and-models.py'
_spec = importlib.util.spec_from_file_location('L3', _p)
_L3 = importlib.util.module_from_spec(_spec)
sys.modules['L3'] = _L3
_spec.loader.exec_module(_L3)

print(f'Setup OK. Lesson module: {_L3.LESSON_NUMBER}')
print(f'Root: {ROOT}')

Setup OK. Lesson module: M3.L3
Root: /Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-course-public/course/ai-native-de


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L3, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 24 tests...

  PASS  test_benchmark_encode_returns_stats
  PASS  test_benchmark_encode_within_budget
  PASS  test_benchmark_includes_warmup
  PASS  test_benchmark_retrieve_scales_with_corpus
  PASS  test_build_dual_index_default_traffic_share
  PASS  test_build_dual_index_returns_routable_object
  PASS  test_build_dual_index_seeds_traffic_rng
  PASS  test_drift_eval_returns_bounded_cosine
  PASS  test_dual_index_query_works
  PASS  test_evaluate_model_returns_metrics
  PASS  test_model_id_is_immutable
  PASS  test_model_normalized
  PASS  test_model_v1_is_cached
  PASS  test_model_v2_is_cached
  PASS  test_monthly_latency_budget_basic
  PASS  test_monthly_latency_budget_scales_linearly
  PASS  test_recall_at_k_partial
  PASS  test_recall_at_k_perfect
  PASS  test_recall_at_k_zero
  PASS  test_retrieve_returns_top_k
  PASS  test_throughput_per_second_increases_as_latency_drops
  PASS  test_throughput_per_second_inverse_of_latency
  PASS  test_two_models_have_different_spaces
  P

In [3]:
# === Section 1: model registry ===
v1 = _L3.model_v1(dim=16)
v2 = _L3.model_v2(dim=16)
print(f'v1: {v1}')
print(f'v2: {v2}')
print(f'Same dim? {v1.dim == v2.dim}  (v1={v1.dim}, v2={v2.dim})')
print(f'Same model_id? {v1.model_id == v2.model_id}')
import numpy as np
print(f'v1 norm check: {float(np.linalg.norm(v1.encode("hello"))):.3f}  (should be 1.0)')

v1: EmbeddingModel(fake-embed@1.0.0, dim=16, normed=True)
v2: EmbeddingModel(fake-embed@2.0.0, dim=16, normed=True)
Same dim? True  (v1=16, v2=16)
Same model_id? True
v1 norm check: 1.000  (should be 1.0)


In [4]:
# === Section 2: corpus + golden queries ===
print(f'Corpus: {len(_L3.SAMPLE_DOCS)} docs')
for d in _L3.SAMPLE_DOCS[:3]:
    print(f'  {d.doc_id}: {d.text[:60]!r}')
print(f'\nGolden queries: {len(_L3.GOLDEN_QUERIES)}')
for g in _L3.GOLDEN_QUERIES[:3]:
    print(f'  {g.query!r}  ->  {g.relevant_doc_ids}')

Corpus: 12 docs
  d01: 'Employee vacation policy: 20 days per year, accrued monthly'
  d02: '401k match: 100% of the first 3% of salary, 50% of the next '
  d03: "Sick leave: 10 days per year, no doctor's note required for "

Golden queries: 12
  'how much vacation do I get'  ->  ['d01']
  'what is the 401k match policy'  ->  ['d02']
  "sick leave and doctor's note policy"  ->  ['d03']


In [5]:
# === Section 3: v1 retrieval on first 3 golden queries ===
v1_ids, v1_matrix = _L3.embed_corpus(v1, _L3.SAMPLE_DOCS)
print('v1 retrieval (top-3):\n')
for g in _L3.GOLDEN_QUERIES[:3]:
    hits = _L3.retrieve(v1, v1_ids, v1_matrix, g.query, k=3)
    top_ids = [h[0] for h in hits]
    r = _L3.recall_at_k(top_ids, g.relevant_doc_ids, k=3)
    print(f'  {g.query!r:<48}  top-3={top_ids}  recall@3={r:.2f}')

v1 retrieval (top-3):

  'how much vacation do I get'                      top-3=['d08', 'd12', 'd02']  recall@3=0.00
  'what is the 401k match policy'                   top-3=['d09', 'd03', 'd01']  recall@3=0.00
  "sick leave and doctor's note policy"             top-3=['d04', 'd10', 'd05']  recall@3=0.00


In [6]:
# === Section 4: v2 retrieval on the same queries ===
v2_ids, v2_matrix = _L3.embed_corpus(v2, _L3.SAMPLE_DOCS)
print('v2 retrieval (top-3):\n')
for g in _L3.GOLDEN_QUERIES[:3]:
    hits = _L3.retrieve(v2, v2_ids, v2_matrix, g.query, k=3)
    top_ids = [h[0] for h in hits]
    r = _L3.recall_at_k(top_ids, g.relevant_doc_ids, k=3)
    print(f'  {g.query!r:<48}  top-3={top_ids}  recall@3={r:.2f}')

v2 retrieval (top-3):

  'how much vacation do I get'                      top-3=['d03', 'd04', 'd09']  recall@3=0.00
  'what is the 401k match policy'                   top-3=['d05', 'd08', 'd04']  recall@3=0.00
  "sick leave and doctor's note policy"             top-3=['d03', 'd04', 'd08']  recall@3=1.00


In [7]:
# === Section 5: drift eval between v1 and v2 ===
d = _L3.drift_eval(v1, v2, _L3.SAMPLE_DOCS)
print(f'mean cosine = {d["mean_cosine_same_text"]:.3f}  (1.0 = identical)')
print(f'min  cosine = {d["min_cosine_same_text"]:.3f}')
print(f'max  cosine = {d["max_cosine_same_text"]:.3f}')
print(f'threshold   = {_L3.DRIFT_THRESHOLD}  (cosine < this => meaningful shift)')
safe = d['mean_cosine_same_text'] >= _L3.DRIFT_THRESHOLD
print(f'safe to roll out? {safe}')

mean cosine = 0.742  (1.0 = identical)
min  cosine = 0.628
max  cosine = 0.829
threshold   = 0.7  (cosine < this => meaningful shift)
safe to roll out? True


In [8]:
# === Section 6: full evaluation on the golden set ===
print('Mean recall@3 over the full golden set:\n')
for label, m in (('v1', v1), ('v2', v2)):
    metrics = _L3.evaluate_model(m, _L3.SAMPLE_DOCS, _L3.GOLDEN_QUERIES, k=3)
    print(f'  {label:<4}  mean_recall@3 = {metrics["mean_recall_at_k"]:.3f}  '
          f'(min={metrics["min_recall_at_k"]:.2f}, n={int(metrics["queries_evaluated"])})')

Mean recall@3 over the full golden set:

  v1    mean_recall@3 = 0.417  (min=0.00, n=12)
  v2    mean_recall@3 = 0.333  (min=0.00, n=12)


In [9]:
# === Section 7: dual-index traffic split ===
di = _L3.DualIndex(
    primary_model=v1, primary_ids=v1_ids, primary_matrix=v1_matrix,
    secondary_model=v2, secondary_ids=v2_ids, secondary_matrix=v2_matrix,
    secondary_traffic_share=0.0,
)
print('Dual-index traffic split (over 1000 requests per share):\n')
rng = np.random.default_rng(seed=42)
for share in (0.0, 0.1, 0.5, 1.0):
    di.secondary_traffic_share = share
    primary_count = secondary_count = 0
    for _ in range(1000):
        if rng.random() < share:
            secondary_count += 1
        else:
            primary_count += 1
    print(f'  share {share:>4.0%}  ->  primary={primary_count:>4},  secondary={secondary_count:>4}')

Dual-index traffic split (over 1000 requests per share):

  share   0%  ->  primary=1000,  secondary=   0
  share  10%  ->  primary= 909,  secondary=  91
  share  50%  ->  primary= 494,  secondary= 506
  share 100%  ->  primary=   0,  secondary=1000


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[corpus text] --> B[v1 model]
    A --> C[v2 model]
    B --> D[v1 index HNSW]
    C --> E[v2 index HNSW]
    F[query] --> G{traffic share}
    G -->|99%| D
    G -->|1%| E
    D --> H[results v1]
    E --> I[results v2]
    H --> J[shadow compare on golden set]
    I --> J
    J -->|drift < threshold| K[don't roll out]
    J -->|drift OK| L[ramp: 1% -> 10% -> 50% -> 100%]
```

**Read as:** Two indexes side by side. Shadow-compare on a golden set. Only ramp if the new model's recall beats or matches the old.

## Reflect (10 min)

1. Why are embeddings model-specific? What breaks if you mix two models in one index?
2. What is drift, and what's a safe threshold for rolling out a new model?
3. Why is dual-index safer than a one-shot switch? What's the worst case without it?
4. In the demo, v1 and v2 drift is near zero. What would real production drift look like, and why?
5. (Stretch) Implement a `drift_alert(drift, threshold)` function that returns a structured warning.

---

**Read next:**
- `lessons/m04-ann/L4-ann-internals.ipynb` — HNSW, IVF-PQ, FAISS benchmarks
- `lessons/m05-ingestion/L5-document-ingestion.ipynb` — idempotent ingestion, content-hash dedupe
- `SYLLABUS.md` — the 18 modules and 21-day plan